# 02 · Motion models and temporal evaluation

**NFL Player Trajectory Lab** · Reproducible real-data benchmark

Compare five physical references, the role-conditioned ridge baseline, and three
feature-family residual challengers. The landing-aware challenger is the development
choice; the original baseline is preserved below for context. The ridge model
learns six shared x/y vector weights for each role, using only training-game sufficient
statistics. Fixed regularization is declared in the protocol before evaluation.

This is a local experiment, not a Kaggle leaderboard score or a medal claim.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
LOCAL = ROOT / "artifacts" / "benchmark"
PUBLISHED = ROOT / "docs" / "results"
RESULTS = LOCAL if (LOCAL / "summary.json").exists() else PUBLISHED
READY = (RESULTS / "summary.json").exists()
if READY:
    summary = json.loads((RESULTS / "summary.json").read_text())
    eda = json.loads((RESULTS / "eda.json").read_text())
    protocol = json.loads((RESULTS / "protocol.json").read_text())
    display(Markdown("**Report source:** " + ("local benchmark artifacts" if RESULTS == LOCAL else "published reproducible experiment snapshot")))
else:
    display(Markdown(
        "No real-data benchmark is available. Run `nfl benchmark` after the audit."
    ))

## The official metric

$$\mathrm{RMSE}=\sqrt{\frac{\sum_{i=1}^{N}[(\hat{x}_i-x_i)^2+(\hat{y}_i-y_i)^2]}{2N}}.$$

Coordinates are measured in yards. Every coordinate/frame receives equal weight.
ADE measures Euclidean displacement, FDE measures each trajectory's final displacement,
and p95 reports the error tail. They complement RMSE; they are different quantities.

In [ ]:
if READY:
    scores = pd.DataFrame(summary["models"])
    display(scores[["model", "coordinate_rmse_yards", "rmse_ci95_low", "rmse_ci95_high", "ade_frame_weighted_yards", "fde_trajectory_weighted_yards", "p95_displacement_yards"]].round(4))
    best = scores.iloc[0]
    display(Markdown(
        f"**Best original baseline:** `{best['model']}`. "
        f"RMSE reduction vs velocity: **{summary['improvement_vs_velocity_percent']:.1f}%**."
    ))

In [ ]:
if READY:
    display(Image(filename=str(RESULTS / "benchmark.png")))

## Interpret the learned weights

The six basis vectors are last velocity × time, recent velocity × time,
acceleration × time²/2, and the vector to the landing point multiplied by normalized
time, normalized time², and normalized time³. Training-RMS scaling and ridge stabilize
the fit. Coordinates share coefficients, preserving rotation and translation equivariance.
An unseen role uses the global training fit. Correlated weights should be interpreted jointly.

In [ ]:
if READY:
    display(Image(filename=str(RESULTS / "coefficients.png")))
    latency = json.loads((RESULTS / "latency.json").read_text())
    display(pd.DataFrame([latency]))

## Watch trajectories and reproduce the experiment

Open `artifacts/benchmark/report.html` for the full offline report and animated field.
The example is the first validation play by game/play ID, selected independently of its errors.

A repeat of `nfl benchmark` verifies cached files and reuses completed weekly stages.
Changed inputs, model source, or dependencies invalidate corresponding checkpoints.
Interrupted stages are recomputed, and a corrupt artifact never counts as completed.

## What this result establishes

Game-cluster bootstrap intervals resample entire games 2,000 times, preserving player
and frame dependence within each sampled game. Paired intervals compare against
constant velocity. They describe this validation sample, not every future season.
Repeated model selection can overfit validation. Keep the holdout reserved until
model selection is locked.

## Where the baseline still struggles

Read the role and forecast-time slices rather than only the overall average.
These are development-validation diagnostics, not new holdout results. Longer
forecasts and defensive coverage motivate the interaction feature experiment.

In [ ]:
if READY:
    slices = pd.DataFrame(summary["slices"])
    display(slices.loc[slices["model"].eq("role_ridge"), ["dimension", "value", "rows", "coordinate_rmse_yards"]].round(4))

## Feature-family ablations

The completed `nfl features` experiment preserves the original model and
prepares one week at a time, screens/scales on training games only, and evaluates
all challengers on exactly the same validation rows. Completed stages are hashed
and reused. The chart includes game-cluster confidence intervals; negative paired
RMSE differences versus role ridge favour a challenger.

The residual learner is intentionally interpretable. It is **not** a reproduced
winning temporal neural model. The result below measures actual development improvement;
feature count and tests alone would not establish that improvement.

In [ ]:
import io

import matplotlib
import numpy as np

matplotlib.use("Agg")
import matplotlib.pyplot as plt

feature_local = ROOT / "artifacts/features/summary.json"
feature_public = PUBLISHED / "feature_summary.json"
feature_path = feature_local if feature_local.is_file() else feature_public
FEATURE_READY = feature_path.is_file()
if FEATURE_READY:
    feature_summary = json.loads(feature_path.read_text())
    assert feature_summary["status"] == "passed"
    assert feature_summary["screening_split"] == "train"
    assert feature_summary["holdout_evaluation"] == "not_run"
    assert feature_summary["split_sha256"] == summary["split_sha256"]
    feature_scores = pd.DataFrame(feature_summary["models"])
    assert np.isfinite(feature_scores["coordinate_rmse_yards"]).all()
    label = "Local experiment" if feature_path == feature_local else "Published experiment snapshot"
    display(Markdown(f"**Evidence:** {label}. **Candidates:** {feature_summary['candidate_features']:,}. " f"**Same validation population:** {feature_summary['validation_games']} games, " f"{feature_summary['validation_rows_per_model']:,} player-frames."))
    display(feature_scores[["model", "selected_features", "coordinate_rmse_yards", "ade_frame_weighted_yards", "fde_trajectory_weighted_yards", "delta_vs_role_ridge_ci95"]].round(4))
    winner = feature_summary["selected_model"]
    scores_by_name = feature_scores.set_index("model")
    gain = 100 * (1 - scores_by_name.loc[winner, "coordinate_rmse_yards"] / scores_by_name.loc["role_ridge", "coordinate_rmse_yards"])
    display(Markdown(
        f"**Choice: `{winner}`.** RMSE reduction vs role ridge: **{gain:.2f}%**. "
        "Validation selection only; not holdout confirmation or a Kaggle score."
    ))
else:
    display(Markdown("No feature experiment is available. The baseline remains the only measured result."))

In [ ]:
if FEATURE_READY:
    ordered = feature_scores.iloc[::-1]
    estimates = ordered["coordinate_rmse_yards"].to_numpy()
    interval = np.vstack([estimates - ordered["rmse_ci95_low"], ordered["rmse_ci95_high"] - estimates])
    fig, ax = plt.subplots(figsize=(10, 4.5), layout="constrained")
    ax.errorbar(estimates, range(len(ordered)), xerr=interval, fmt="o", capsize=5)
    ax.set_yticks(range(len(ordered)), ordered["model"])
    ax.set_xlabel("Coordinate RMSE (yards) · lower is better")
    ax.set_title("Landing-aware features lead the tested ablations", loc="left", pad=14)
    ax.spines[["top", "right"]].set_visible(False)
    fig.supxlabel("95% game-cluster bootstrap intervals; 32 shared validation games", fontsize=9)
    image = io.BytesIO()
    fig.savefig(image, format="png", dpi=150)
    plt.close(fig)
    display(Image(data=image.getvalue()))

## Diagnose what remains difficult

The paired landing-versus-role RMSE interval is entirely below zero on this validation set. It supports a measured development improvement, but it does not account for all future experimentation or replace the held-out games. **The larger interaction-aware candidate set did not beat the landing-aware set in this fixed-budget ridge experiment.** This does not prove player interactions are useless: screening competition, model capacity, and nonlinear relationships can change the outcome.

Defense and longer forecasts are the next error-analysis priorities. Display row counts alongside slices: the fourth-second estimate has only 127 player-frames and should not dominate conclusions.

In [ ]:
if FEATURE_READY:
    slices = pd.DataFrame(feature_summary["slices"])
    chosen = slices[slices["model"].isin([winner, "role_ridge"])]
    role_rows = chosen[chosen["dimension"] == "role"]
    display(role_rows[["model", "value", "rows", "coordinate_rmse_yards"]].round(4))
    horizon = chosen[chosen["dimension"] == "forecast_second"].copy()
    horizon["second"] = horizon["value"].astype(int)
    fig, ax = plt.subplots(figsize=(10, 4.5), layout="constrained")
    for model_name, values in horizon.groupby("model", sort=True):
        values = values.sort_values("second")
        ax.plot(values["second"], values["coordinate_rmse_yards"], marker="o", label=model_name)
    counts = horizon[horizon["model"] == winner].sort_values("second")
    ax.set_xticks(counts["second"], [f"Second {int(row.second)}\nn={int(row.rows):,}" for row in counts.itertuples()])
    ax.set_ylabel("Coordinate RMSE (yards)")
    ax.set_title("Forecast difficulty rises with time in the air", loc="left", pad=14)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    image = io.BytesIO()
    fig.savefig(image, format="png", dpi=150)
    plt.close(fig)
    display(Image(data=image.getvalue()))

## Model decision and next experiment

**Keep the verified role ridge as a reproducible reference; use the landing residual ridge as the development challenger.** Do not add thousands of arbitrary columns simply to increase the count. The next experiment should compare a nonlinear residual model and an interaction-aware temporal encoder against this measured challenger, retaining the same outer temporal split. Fit feature selection and scaling only on training games; tune within training games rather than repeatedly optimizing this validation set. Test landing-relative motion and receiver/defender geometry with controlled ablations.

Holdout scoring stays deferred until the model-selection procedure is frozen. No temporal neural training, optimizer-state recovery, or leaderboard score is claimed here.

## Generate your own inference notebook

The cell below is **off by default**. In your existing SageMaker checkout, set `GENERATE_SUBMISSION = True` and run it to build the standalone inference notebook from your verified saved weights. It does not fit models, contact Kaggle, or upload predictions. Download the generated notebook using its output link.

In Kaggle, import that notebook, attach **NFL Big Data Bowl 2026 Prediction**, use CPU and disable internet. Running all cells executes the official local gateway and produces validated **`submission.parquet`**, with download links. This competition uses an inference API: a preview download is not a scored submission. You decide whether to submit a saved Kaggle version, subject to your account's late-submission eligibility.

Completed local play predictions are checksummed and reusable on the retained filesystem. An interrupted play restarts; a new Kaggle session does not automatically recover an old cache. Hidden reruns do not reuse preview caches.

In [ ]:
GENERATE_SUBMISSION = False
EXPORT_MODEL = "landing_ridge"

if GENERATE_SUBMISSION:
    import base64
    import hashlib
    import runpy

    from IPython.display import HTML

    baseline_path = ROOT / "artifacts/benchmark/model.json"
    feature_model_path = ROOT / "artifacts/features/model.json"
    if not baseline_path.is_file() or not feature_model_path.is_file():
        raise FileNotFoundError(
            "Saved baseline and feature weights are required. No automatic retraining."
        )
    exporter = runpy.run_path(str(ROOT / "kaggle/export.py"))
    generated = exporter["export_notebook"](ROOT, EXPORT_MODEL, baseline_path, feature_model_path)
    manifest_path = generated.parent / "export_manifest.json"
    manifest = json.loads(manifest_path.read_text())
    assert manifest["model"] == EXPORT_MODEL
    assert manifest["notebook_sha256"] == hashlib.sha256(generated.read_bytes()).hexdigest()
    assert manifest["uploaded_to_kaggle"] is False
    for path in (generated, manifest_path):
        payload = base64.b64encode(path.read_bytes()).decode("ascii")
        display(HTML(
            f'<a download="{path.name}" '
            f'href="data:application/octet-stream;base64,{payload}">'
            f'Download your generated {path.name}</a>'
        ))
    print("EXPORT_VERIFIED · saved weights only · no upload · official gateway must be run in Kaggle")
else:
    print(
        "Generation is off. Set GENERATE_SUBMISSION = True to create your inference notebook."
    )